# 📄 Marker-PDF API Server — Colab
> شغّل الـ cells بالترتيب، وبعدين انسخ الـ ngrok URL وحطّه في `pdf_agent_suite.py`

---
### خطوات:
1. شغّل **Cell 1** — تثبيت المكتبات
2. شغّل **Cell 2** — حط ngrok token بتاعك
3. شغّل **Cell 3** — تشغيل السيرفر
4. انسخ الـ URL الظاهر وحطّه في `MARKER_API_URL` في السكريبت

---
## 🔧 Cell 1 — تثبيت المكتبات

In [ ]:
!pip install marker-pdf fastapi uvicorn python-multipart pyngrok nest-asyncio -q

import importlib
for lib in ['marker', 'fastapi', 'uvicorn', 'pyngrok', 'nest_asyncio']:
    try:
        importlib.import_module(lib)
        print(f'  ✅ {lib}')
    except ImportError:
        print(f'  ❌ {lib} — فشل التثبيت!')

print('\n✅ التثبيت اكتمل!')

---
## ⚙️ Cell 2 — إعداد ngrok
> احصل على token مجاني من: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"   # ← ✏️ حط الـ token بتاعك هنا

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token اتحفظ!')

---
## 🚀 Cell 3 — تشغيل السيرفر
> هيظهر الـ ngrok URL — انسخه وحطّه في `MARKER_API_URL` في السكريبت بتاعك

In [ ]:
import os, tempfile, threading, nest_asyncio
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="Marker-PDF Server")

# ── تحميل الموديل ─────────────────────────────────────────
print('⏳ Loading marker-pdf models ...')
_converter = None

try:
    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    from marker.config.parser import ConfigParser

    _config = {
        'output_format'              : 'markdown',
        'langs'                      : ['ar', 'en'],
        'force_ocr'                  : False,
        'ocr_all_pages'              : False,
        'strip_existing_ocr'         : False,
        'extract_images'             : False,
        'image_descriptive_captions' : False,
    }
    _cfg_parser = ConfigParser(_config)
    _model_dict = create_model_dict()   # Colab GPU تلقائياً
    _converter  = PdfConverter(
        config        = _cfg_parser.generate_config_dict(),
        artifact_dict = _model_dict,
        processor_list= None,
        renderer      = None,
    )
    print('✅ Marker-PDF models loaded!')

except Exception as e:
    print(f'❌ فشل تحميل marker-pdf: {e}')


# ── Endpoints ─────────────────────────────────────────────
@app.get('/')
def root():
    return {'status': 'ok', 'service': 'marker-pdf-server'}


@app.get('/health')
def health():
    return {'status': 'ok', 'model_loaded': _converter is not None}


@app.post('/extract')
async def extract_pdf(file: UploadFile = File(...)):
    if _converter is None:
        raise HTTPException(status_code=503, detail='Marker model not loaded')
    if not file.filename.lower().endswith('.pdf'):
        raise HTTPException(status_code=400, detail='ارفع ملف PDF بس')

    with tempfile.NamedTemporaryFile(delete=False, suffix='.pdf') as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    try:
        rendered  = _converter(tmp_path)
        full_text = ''
        try:
            from marker.output import text_from_rendered
            result = text_from_rendered(rendered)
            full_text = result[0] if isinstance(result, tuple) else result
        except Exception:
            for attr in ('markdown', 'text', 'output', 'content'):
                val = getattr(rendered, attr, None)
                if val and isinstance(val, str):
                    full_text = val
                    break

        return JSONResponse({
            'status'    : 'ok',
            'filename'  : file.filename,
            'characters': len(full_text),
            'markdown'  : full_text,
        })
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        os.unlink(tmp_path)


# ── تشغيل ngrok + السيرفر ──────────────────────────────────
PORT = 7070
public_url = ngrok.connect(PORT)

print('\n' + '='*55)
print('🌐 MARKER SERVER URL (انسخه وحطّه في السكريبت):')
print(f'   {public_url}')
print('='*55)
print(f'\nفي pdf_agent_suite.py غيّر السطر ده:')
print(f'   MARKER_API_URL = "{public_url}"')
print('\n⏳ السيرفر شغّال ... (لا تقفل الـ cell)')

uvicorn.run(app, host='0.0.0.0', port=PORT)